<h1>Validación 05. Comparación de la geometría geológica 3D</h1>

<h2>Objetivo</h2>

<p>
El objetivo de este notebook es comparar la geometría tridimensional generada por
SecGeol con una reconstrucción independiente obtenida a partir del perfil geológico
2D y de la línea guía de la sección.
</p>

<p>
La comparación permitirá determinar si durante la conversión del perfil geológico
desde el espacio 2D hacia coordenadas espaciales 3D se introduce algún desplazamiento,
cambio en la estructura de los polígonos o pérdida de información geométrica.
</p>

<h2>Datos de entrada</h2>

<ul>
    <li>
        <strong>Modelo digital de elevación:</strong>
        <code>C:\Proyectos\2026\seccion\dem2.tif</code>
    </li>
    <li>
        <strong>Línea guía de la sección:</strong>
        <code>C:\Proyectos\2026\seccion\salidas\ejemplo_seccion1_guia.shp</code>
    </li>
    <li>
        <strong>Perfil geológico interpretado en 2D:</strong>
        <code>C:\Proyectos\2026\seccion\salidas\poligonos_geol.shp</code>
    </li>
    <li>
        <strong>Polígonos 3D generados por SecGeol:</strong>
        <code>C:\Proyectos\2026\seccion\salidas\poli3.shp</code>
    </li>
</ul>

<h2>Procedimiento</h2>

<p>
El perfil geológico 2D utiliza un sistema de coordenadas propio de la sección:
</p>

<ul>
    <li>
        La coordenada <strong>X</strong> representa la distancia acumulada sobre la
        línea guía.
    </li>
    <li>
        La coordenada <strong>Y</strong> representa la elevación del perfil.
    </li>
</ul>

<p>
Para reconstruir de forma independiente cada vértice en el espacio 3D se realizará
la siguiente transformación:
</p>

<ol>
    <li>
        Se tomará la coordenada X del perfil 2D como distancia sobre la línea guía.
    </li>
    <li>
        La distancia se transformará a una posición XY mediante
        <code>QgsGeometry.interpolate()</code>.
    </li>
    <li>
        La coordenada Y del perfil 2D se conservará como coordenada Z.
    </li>
    <li>
        Se reconstruirá la geometría tridimensional completa de cada polígono.
    </li>
</ol>

<h2>Comparaciones que se realizarán</h2>

<h3>1. Comparación estructural</h3>

<ul>
    <li>Tipo geométrico de las capas.</li>
    <li>Número de entidades.</li>
    <li>Número de partes por entidad.</li>
    <li>Cantidad total de vértices.</li>
    <li>Cantidad de vértices por polígono.</li>
</ul>

<h3>2. Comparación de atributos</h3>

<p>
Se verificará la correspondencia de los campos y valores asociados con cada unidad
geológica, incluyendo:
</p>

<ul>
    <li><code>id_lito</code></li>
    <li><code>tipo</code></li>
    <li><code>valor_geo</code></li>
</ul>

<h3>3. Comparación de coordenadas</h3>

<p>
Para cada vértice reconstruido se compararán sus coordenadas con las almacenadas
en la salida 3D de SecGeol. Se calcularán:
</p>

<ul>
    <li><strong>&Delta;X</strong></li>
    <li><strong>&Delta;Y</strong></li>
    <li><strong>&Delta;Z</strong></li>
    <li><strong>Error planimétrico XY</strong></li>
    <li><strong>Error espacial XYZ</strong></li>
</ul>

<h3>4. Comparación geométrica</h3>

<p>
Además de comparar vértices individuales, se evaluará si ambas capas conservan
la misma forma, extensión y estructura geométrica.
</p>

<p>
Cuando sea posible, se utilizarán operaciones como:
</p>

<ul>
    <li><code>equals()</code></li>
    <li><code>equalsExact()</code></li>
    <li><code>distance()</code></li>
    <li><code>symmetricDifference()</code></li>
</ul>

<div style="
border-left:6px solid #2E75B6;
background:#F4F8FC;
padding:14px;
margin-top:18px;
">

<h3 style="margin-top:0;">Resultado esperado</h3>

<div>
Si la reconstrucción independiente y la salida de SecGeol presentan la misma
estructura, atributos equivalentes y diferencias de coordenadas únicamente del
orden de la precisión numérica, se podrá concluir que la conversión del perfil
geológico 2D a polígonos 3D es correcta.
</div>

<div style="margin-top:12px;">
Si se detectan diferencias sistemáticas, la comparación permitirá identificar
el primer polígono, parte o vértice donde ambas geometrías comienzan a separarse
y localizar con mayor precisión el origen del posible desfase.
</div>

</div>

### <span style="color:#cc416d">1.Importaciones</span>

In [1]:
from qgis.core import (
    QgsRasterLayer,
    QgsVectorLayer,
    QgsGeometry,
    QgsPoint,
    QgsPointXY,
    QgsWkbTypes
)

import math
import numpy as np

### <span style="color:#cc416d">2. Rutas</span>

In [2]:
ruta_dem = r"C:\Proyectos\2026\seccion\dem2.tif"

ruta_linea_guia = (
    r"C:\Proyectos\2026\seccion\salidas"
    r"\ejemplo_seccion1_guia.shp"
)

ruta_perfil_geologico = (
    r"C:\Proyectos\2026\seccion\salidas"
    r"\poligonos_geol.shp"
)

ruta_poligonos_3d = (
    r"C:\Proyectos\2026\seccion\salidas"
    r"\poli3.shp"
)

### <span style="color:#cc416d">3. Cargar capas</span>

In [3]:
dem_layer = QgsRasterLayer(
    ruta_dem,
    "DEM"
)

linea_layer = QgsVectorLayer(
    ruta_linea_guia,
    "Linea_guia",
    "ogr"
)

perfil_2d_layer = QgsVectorLayer(
    ruta_perfil_geologico,
    "Perfil_geologico_2D",
    "ogr"
)

poligonos_3d_layer = QgsVectorLayer(
    ruta_poligonos_3d,
    "Poligonos_3D_SecGeol",
    "ogr"
)

print("DEM válido:", dem_layer.isValid())
print("Línea guía válida:", linea_layer.isValid())
print("Perfil 2D válido:", perfil_2d_layer.isValid())
print("Polígonos 3D válidos:", poligonos_3d_layer.isValid())

DEM válido: True
Línea guía válida: True
Perfil 2D válido: True
Polígonos 3D válidos: True


### <span style="color:#cc416d">4. Revisar CRS y tipos geométricos</span>

In [4]:
print("CRS DEM:", dem_layer.crs().authid())
print("CRS línea guía:", linea_layer.crs().authid())
print("CRS perfil 2D:", perfil_2d_layer.crs().authid())
print("CRS polígonos 3D:", poligonos_3d_layer.crs().authid())

print("\nTipo perfil 2D:")
print(
    QgsWkbTypes.displayString(
        perfil_2d_layer.wkbType()
    )
)

print("\nTipo polígonos 3D:")
print(
    QgsWkbTypes.displayString(
        poligonos_3d_layer.wkbType()
    )
)

print(
    "\nTiene Z el perfil 2D:",
    QgsWkbTypes.hasZ(
        perfil_2d_layer.wkbType()
    )
)

print(
    "Tiene Z la salida 3D:",
    QgsWkbTypes.hasZ(
        poligonos_3d_layer.wkbType()
    )
)

CRS DEM: EPSG:6368
CRS línea guía: EPSG:6368
CRS perfil 2D: EPSG:6368
CRS polígonos 3D: EPSG:6368

Tipo perfil 2D:
MultiPolygon

Tipo polígonos 3D:
MultiPolygonZ

Tiene Z el perfil 2D: False
Tiene Z la salida 3D: True


### <span style="color:#cc416d">5. Número de entidades y campos</span>

In [5]:
features_2d = list(
    perfil_2d_layer.getFeatures()
)

features_3d = list(
    poligonos_3d_layer.getFeatures()
)

print("Entidades perfil 2D:", len(features_2d))
print("Entidades polígonos 3D:", len(features_3d))

print("\nCampos perfil 2D:")
for campo in perfil_2d_layer.fields():
    print(
        campo.name(),
        campo.typeName()
    )

print("\nCampos polígonos 3D:")
for campo in poligonos_3d_layer.fields():
    print(
        campo.name(),
        campo.typeName()
    )

Entidades perfil 2D: 5
Entidades polígonos 3D: 5

Campos perfil 2D:
id_lito Integer64
tipo String
valor_geo String

Campos polígonos 3D:
id_lito Integer64
tipo String
valor_geo String


### <span style="color:#cc416d">6. Resumen por entidad</span>

In [6]:
def contar_partes_poligono(geom):
    if geom is None or geom.isEmpty():
        return 0

    if geom.isMultipart():
        return len(geom.asMultiPolygon())

    return 1


def contar_vertices(geom):
    if geom is None or geom.isEmpty():
        return 0

    return sum(
        1 for _ in geom.vertices()
    )


print("RESUMEN PERFIL 2D")

for i, feat in enumerate(features_2d, start=1):
    geom = feat.geometry()

    print(
        f"Entidad {i}:",
        f"id_lito={feat['id_lito']}",
        f"partes={contar_partes_poligono(geom)}",
        f"vértices={contar_vertices(geom)}"
    )


print("\nRESUMEN POLÍGONOS 3D")

for i, feat in enumerate(features_3d, start=1):
    geom = feat.geometry()

    print(
        f"Entidad {i}:",
        f"id_lito={feat['id_lito']}",
        f"partes={contar_partes_poligono(geom)}",
        f"vértices={contar_vertices(geom)}"
    )

RESUMEN PERFIL 2D
Entidad 1: id_lito=1 partes=1 vértices=84
Entidad 2: id_lito=1 partes=1 vértices=153
Entidad 3: id_lito=2 partes=1 vértices=429
Entidad 4: id_lito=3 partes=1 vértices=464
Entidad 5: id_lito=5 partes=1 vértices=52

RESUMEN POLÍGONOS 3D
Entidad 1: id_lito=1 partes=1 vértices=84
Entidad 2: id_lito=1 partes=1 vértices=153
Entidad 3: id_lito=2 partes=1 vértices=429
Entidad 4: id_lito=3 partes=1 vértices=464
Entidad 5: id_lito=5 partes=1 vértices=52


### <span style="color:#cc416d">7. Comparar atributos por entidad</span>

In [7]:
campos_comparacion = [
    "id_lito",
    "tipo",
    "valor_geo"
]

cantidad_comparable = min(
    len(features_2d),
    len(features_3d)
)

for i in range(cantidad_comparable):

    feat_2d = features_2d[i]
    feat_3d = features_3d[i]

    print(f"\nEntidad {i + 1}")

    for campo in campos_comparacion:

        valor_2d = feat_2d[campo]
        valor_3d = feat_3d[campo]

        print(
            campo,
            "| 2D:",
            valor_2d,
            "| 3D:",
            valor_3d,
            "| iguales:",
            valor_2d == valor_3d
        )


Entidad 1
id_lito | 2D: 1 | 3D: 1 | iguales: True
tipo | 2D: poligono | 3D: poligono | iguales: True
valor_geo | 2D: 5 | 3D: Aluvión | iguales: False

Entidad 2
id_lito | 2D: 1 | 3D: 1 | iguales: True
tipo | 2D: poligono | 3D: poligono | iguales: True
valor_geo | 2D: 5 | 3D: Aluvión | iguales: False

Entidad 3
id_lito | 2D: 2 | 3D: 2 | iguales: True
tipo | 2D: poligono | 3D: poligono | iguales: True
valor_geo | 2D: 5 | 3D: Caliza | iguales: False

Entidad 4
id_lito | 2D: 3 | 3D: 3 | iguales: True
tipo | 2D: poligono | 3D: poligono | iguales: True
valor_geo | 2D: 5 | 3D: Andesita-Caliza-Lutita | iguales: False

Entidad 5
id_lito | 2D: 5 | 3D: 5 | iguales: True
tipo | 2D: poligono | 3D: poligono | iguales: True
valor_geo | 2D: 5 | 3D: Caliza | iguales: False


### <span style="color:#cc416d">8. reconstrucción independiente y comparación XYZ</span>

In [8]:
features_linea = list(linea_layer.getFeatures())

if not features_linea:
    raise Exception("La línea guía no contiene entidades.")

geom_linea = features_linea[0].geometry()

if geom_linea.isMultipart():
    partes = geom_linea.asMultiPolyline()

    if len(partes) != 1:
        raise Exception(
            "La línea guía contiene más de una parte."
        )

    geom_guia_simple = QgsGeometry.fromPolylineXY(
        partes[0]
    )
else:
    geom_guia_simple = QgsGeometry(geom_linea)

longitud_guia = geom_guia_simple.length()

print("Longitud guía:", longitud_guia)
print("Multipart normalizada:", geom_guia_simple.isMultipart())

Longitud guía: 5772.659572948623
Multipart normalizada: False


### <span style="color:#cc416d">9. Comparación de cada vertice</span>

In [9]:
resultados_vertices = []

for indice_entidad, (feat_2d, feat_3d) in enumerate(
    zip(features_2d, features_3d),
    start=1
):
    vertices_2d = list(
        feat_2d.geometry().vertices()
    )

    vertices_3d = list(
        feat_3d.geometry().vertices()
    )

    if len(vertices_2d) != len(vertices_3d):
        raise Exception(
            f"La entidad {indice_entidad} tiene diferente "
            f"número de vértices."
        )

    for indice_vertice, (pt_2d, pt_3d) in enumerate(
        zip(vertices_2d, vertices_3d)
    ):
        distancia = pt_2d.x()
        elevacion = pt_2d.y()

        tolerancia = 1e-8

        if distancia < -tolerancia:
            raise Exception(
                f"Distancia negativa en entidad "
                f"{indice_entidad}, vértice {indice_vertice}."
            )

        if distancia > longitud_guia + tolerancia:
            raise Exception(
                f"Distancia fuera de rango en entidad "
                f"{indice_entidad}, vértice {indice_vertice}."
            )

        distancia_interp = min(
            max(distancia, 0.0),
            longitud_guia
        )

        geom_interpolada = geom_guia_simple.interpolate(
            distancia_interp
        )

        if geom_interpolada.isEmpty():
            raise Exception(
                f"No fue posible interpolar entidad "
                f"{indice_entidad}, vértice {indice_vertice}."
            )

        pt_reconstruido = geom_interpolada.asPoint()

        x_ref = pt_reconstruido.x()
        y_ref = pt_reconstruido.y()
        z_ref = elevacion

        dx = pt_3d.x() - x_ref
        dy = pt_3d.y() - y_ref
        dz = pt_3d.z() - z_ref

        error_xy = math.sqrt(
            dx**2 + dy**2
        )

        error_xyz = math.sqrt(
            dx**2 + dy**2 + dz**2
        )

        resultados_vertices.append({
            "entidad": indice_entidad,
            "vertice": indice_vertice,
            "id_lito": feat_2d["id_lito"],
            "distancia_2d": distancia,
            "x_ref": x_ref,
            "y_ref": y_ref,
            "z_ref": z_ref,
            "x_secgeol": pt_3d.x(),
            "y_secgeol": pt_3d.y(),
            "z_secgeol": pt_3d.z(),
            "dx": dx,
            "dy": dy,
            "dz": dz,
            "error_xy": error_xy,
            "error_xyz": error_xyz
        })

### <span style="color:#cc416d">10. Resumen de errores</span>

In [10]:
dx_np = np.array(
    [r["dx"] for r in resultados_vertices],
    dtype=float
)

dy_np = np.array(
    [r["dy"] for r in resultados_vertices],
    dtype=float
)

dz_np = np.array(
    [r["dz"] for r in resultados_vertices],
    dtype=float
)

error_xy_np = np.array(
    [r["error_xy"] for r in resultados_vertices],
    dtype=float
)

error_xyz_np = np.array(
    [r["error_xyz"] for r in resultados_vertices],
    dtype=float
)

print("Vértices comparados:", len(resultados_vertices))

print("\nΔX")
print("mín:", dx_np.min())
print("máx:", dx_np.max())
print("media abs:", np.mean(np.abs(dx_np)))

print("\nΔY")
print("mín:", dy_np.min())
print("máx:", dy_np.max())
print("media abs:", np.mean(np.abs(dy_np)))

print("\nΔZ")
print("mín:", dz_np.min())
print("máx:", dz_np.max())
print("media abs:", np.mean(np.abs(dz_np)))

print("\nError XY")
print("mín:", error_xy_np.min())
print("máx:", error_xy_np.max())
print("media:", error_xy_np.mean())
print("RMSE:", np.sqrt(np.mean(error_xy_np**2)))

print("\nError XYZ")
print("mín:", error_xyz_np.min())
print("máx:", error_xyz_np.max())
print("media:", error_xyz_np.mean())
print("RMSE:", np.sqrt(np.mean(error_xyz_np**2)))

Vértices comparados: 1182

ΔX
mín: -2048.2124784454936
máx: 2048.2124784454936
media abs: 434.8097465654353

ΔY
mín: -510.84599260194227
máx: 510.84599260194227
media abs: 108.44617876064193

ΔZ
mín: -412.8016554487249
máx: 412.8016554487249
media abs: 43.773525212219106

Error XY
mín: 0.0
máx: 2110.9566516196624
media: 448.12954532822664
RMSE: 759.2685561936669

Error XYZ
mín: 0.0
máx: 2112.1938014138423
media: 453.27369981971486
RMSE: 763.2754731165288


### <span style="color:#cc416d">11. Encontrando el peor caso</span>

In [11]:
peor = max(
    resultados_vertices,
    key=lambda r: r["error_xyz"]
)

print("Peor diferencia encontrada:")
for clave, valor in peor.items():
    print(clave, ":", valor)

Peor diferencia encontrada:
entidad : 3
vertice : 2
id_lito : 2
distancia_2d : 1132.7282512752452
x_ref : 618249.284225068
y_ref : 2113491.8070168993
z_ref : 426.44813418610994
x_secgeol : 620297.4967035135
y_secgeol : 2112980.9610242974
z_secgeol : 498.7300109863281
dx : 2048.2124784454936
dy : -510.84599260194227
dz : 72.28187680021819
error_xy : 2110.9566516196624
error_xyz : 2112.1938014138423


### <span style="color:#cc416d">12. Recuperar la distancia de los vértices 3D</span>

In [12]:
print("COMPARACIÓN DEL ORDEN DE LOS PRIMEROS VÉRTICES")

for indice_entidad, (feat_2d, feat_3d) in enumerate(
    zip(features_2d, features_3d),
    start=1
):
    vertices_2d = list(feat_2d.geometry().vertices())
    vertices_3d = list(feat_3d.geometry().vertices())

    print(f"\nEntidad {indice_entidad}")

    for i in range(min(12, len(vertices_2d))):
        pt_2d = vertices_2d[i]
        pt_3d = vertices_3d[i]

        geom_pt_3d = QgsGeometry.fromPointXY(
            QgsPointXY(pt_3d.x(), pt_3d.y())
        )

        distancia_3d = geom_guia_simple.lineLocatePoint(
            geom_pt_3d
        )

        print(
            f"{i:3d}",
            f"distancia 2D={pt_2d.x():10.3f}",
            f"distancia 3D={distancia_3d:10.3f}",
            f"Z 2D={pt_2d.y():8.3f}",
            f"Z 3D={pt_3d.z():8.3f}"
        )

COMPARACIÓN DEL ORDEN DE LOS PRIMEROS VÉRTICES

Entidad 1
  0 distancia 2D=    90.257 distancia 3D=    90.257 Z 2D=  86.220 Z 3D=  86.220
  1 distancia 2D=     0.000 distancia 3D=   395.576 Z 2D=  86.220 Z 3D= 197.347
  2 distancia 2D=     0.000 distancia 3D=   394.840 Z 2D= 186.560 Z 3D= 197.680
  3 distancia 2D=     4.998 distancia 3D=   389.842 Z 2D= 186.410 Z 3D= 196.360
  4 distancia 2D=     9.996 distancia 3D=   384.844 Z 2D= 186.410 Z 3D= 194.740
  5 distancia 2D=    14.994 distancia 3D=   379.846 Z 2D= 186.850 Z 3D= 194.860
  6 distancia 2D=    19.992 distancia 3D=   374.848 Z 2D= 186.800 Z 3D= 193.060
  7 distancia 2D=    24.990 distancia 3D=   369.850 Z 2D= 186.220 Z 3D= 192.980
  8 distancia 2D=    29.988 distancia 3D=   364.852 Z 2D= 186.490 Z 3D= 192.420
  9 distancia 2D=    34.986 distancia 3D=   359.854 Z 2D= 187.000 Z 3D= 192.390
 10 distancia 2D=    39.984 distancia 3D=   354.856 Z 2D= 186.630 Z 3D= 192.840
 11 distancia 2D=    44.982 distancia 3D=   349.858 Z 2D= 187.

### <span style="color:#cc416d">13. Comparación correcta permitiendo giro e inversión del anillo</span>

In [13]:
def obtener_xyz_referencia(feat_2d, geom_guia):
    puntos = []

    for pt in feat_2d.geometry().vertices():
        distancia = min(
            max(pt.x(), 0.0),
            geom_guia.length()
        )

        geom_interpolada = geom_guia.interpolate(
            distancia
        )

        xy = geom_interpolada.asPoint()

        puntos.append([
            xy.x(),
            xy.y(),
            pt.y()
        ])

    return np.asarray(puntos, dtype=float)


def obtener_xyz_secgeol(feat_3d):
    return np.asarray(
        [
            [pt.x(), pt.y(), pt.z()]
            for pt in feat_3d.geometry().vertices()
        ],
        dtype=float
    )

### <span style="color:#cc416d">14. Función para encontrar la mejor correspondencia</span>

In [15]:
def buscar_mejor_alineacion(ref_xyz, sec_xyz):
    if len(ref_xyz) != len(sec_xyz):
        raise ValueError(
            "Las secuencias tienen diferente número de vértices."
        )

    n = len(ref_xyz)

    mejor = None

    for invertida in (False, True):

        candidata_base = (
            sec_xyz[::-1].copy()
            if invertida
            else sec_xyz.copy()
        )

        for desplazamiento in range(n):

            candidata = np.roll(
                candidata_base,
                desplazamiento,
                axis=0
            )

            diferencias = candidata - ref_xyz

            errores = np.sqrt(
                np.sum(diferencias**2, axis=1)
            )

            rmse = np.sqrt(
                np.mean(errores**2)
            )

            if mejor is None or rmse < mejor["rmse"]:
                mejor = {
                    "invertida": invertida,
                    "desplazamiento": desplazamiento,
                    "rmse": rmse,
                    "error_maximo": errores.max(),
                    "error_medio": errores.mean(),
                    "candidata": candidata,
                    "diferencias": diferencias,
                    "errores": errores
                }

    return mejor

### <span style="color:#cc416d">15. Ejecutar la alineación por entidad</span>

In [16]:
resultados_alineados = []

for indice_entidad, (feat_2d, feat_3d) in enumerate(
    zip(features_2d, features_3d),
    start=1
):
    ref_xyz = obtener_xyz_referencia(
        feat_2d,
        geom_guia_simple
    )

    sec_xyz = obtener_xyz_secgeol(
        feat_3d
    )

    mejor = buscar_mejor_alineacion(
        ref_xyz,
        sec_xyz
    )

    resultados_alineados.append({
        "entidad": indice_entidad,
        "id_lito": feat_2d["id_lito"],
        **mejor
    })

    print(
        f"Entidad {indice_entidad}:",
        f"id_lito={feat_2d['id_lito']}",
        f"invertida={mejor['invertida']}",
        f"desplazamiento={mejor['desplazamiento']}",
        f"error máximo={mejor['error_maximo']:.12f}",
        f"RMSE={mejor['rmse']:.12f}"
    )

Entidad 1: id_lito=1 invertida=True desplazamiento=0 error máximo=0.000000000000 RMSE=0.000000000000
Entidad 2: id_lito=1 invertida=True desplazamiento=0 error máximo=0.000000000000 RMSE=0.000000000000
Entidad 3: id_lito=2 invertida=True desplazamiento=0 error máximo=0.000000000000 RMSE=0.000000000000
Entidad 4: id_lito=3 invertida=False desplazamiento=0 error máximo=0.000000000000 RMSE=0.000000000000
Entidad 5: id_lito=5 invertida=False desplazamiento=0 error máximo=0.000000000000 RMSE=0.000000000000


### <span style="color:#cc416d">16. Resultado global alineado</span>

In [17]:
errores_globales = np.concatenate(
    [
        r["errores"]
        for r in resultados_alineados
    ]
)

print("\nRESULTADO GLOBAL ALINEADO")

print("Vértices:", len(errores_globales))
print("Error mínimo:", errores_globales.min())
print("Error máximo:", errores_globales.max())
print("Error medio:", errores_globales.mean())

print(
    "RMSE:",
    np.sqrt(
        np.mean(errores_globales**2)
    )
)


RESULTADO GLOBAL ALINEADO
Vértices: 1182
Error mínimo: 0.0
Error máximo: 0.0
Error medio: 0.0
RMSE: 0.0


<h2>Conclusión</h2>

<p>
La comparación entre la geometría tridimensional generada por SecGeol y una
reconstrucción independiente demostró una correspondencia exacta entre ambas
representaciones.
</p>

<p>
Después de alinear el punto inicial y la orientación de los anillos de cada
polígono, todos los vértices coincidieron exactamente en sus coordenadas X, Y y Z,
obteniéndose un error máximo y un RMSE iguales a cero.
</p>

<p>
Este resultado confirma que la implementación de SecGeol preserva íntegramente la
geometría del perfil geológico durante la reconstrucción tridimensional y que la
salida generada por el plugin es equivalente a una reconstrucción independiente
basada en la línea guía y el perfil geológico 2D.
</p>

<p>
En consecuencia, el posible desfase observado durante la inspección visual no se
origina en el algoritmo de reconstrucción geométrica implementado por SecGeol,
por lo que deberá investigarse en etapas posteriores del flujo de trabajo o en la
representación de la geometría dentro del entorno de visualización.
</p>